# train_models.ipynb

This notebook trains and stores all models to disk.
- All feature-engineering logic is imported from `utils.py`.
- If the `models/` directory already exists the notebook **will not overwrite** the stored files.
- Results are fully reproducible: run it twice and you get identical output.

**Data path:** `$HOME/Datasets/QuoraQuestionPairs/quora_data.csv`

In [1]:
import os, sys
import pandas as pd
import numpy as np
import sklearn
import sklearn.linear_model
import sklearn.ensemble
import joblib

sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
from utils import (
    get_data_splits,
    fit_count_vectorizer,
    fit_tfidf_vectorizer,
    get_bow_features,
    build_all_features,
    evaluate_model,
    save_model,
)

## 1 – Load data & create splits

In [2]:
DATA_PATH_DEFAULT = os.path.join(os.path.expanduser("~"), "Datasets", "QuoraQuestionPairs", "quora_data.csv")
DATA_PATH_LOCAL = "quora_data.csv"

if os.path.exists(DATA_PATH_DEFAULT):
    DATA_PATH = DATA_PATH_DEFAULT
elif os.path.exists(DATA_PATH_LOCAL):
    DATA_PATH = DATA_PATH_LOCAL
else:
    raise FileNotFoundError(f"Could not find quora_data.csv in {DATA_PATH_DEFAULT} or {DATA_PATH_LOCAL}")

quora_df = pd.read_csv(DATA_PATH)
train_df, val_df, test_df = get_data_splits(quora_df)

print('train_df.shape=', train_df.shape)
print('val_df.shape=',   val_df.shape)
print('test_df.shape=',  test_df.shape)

train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


## 2 – Check if models already exist

In [3]:
MODELS_DIR = "models"

MODELS_EXIST = os.path.isdir(MODELS_DIR) and len(os.listdir(MODELS_DIR)) > 0
if MODELS_EXIST:
    print("models/ directory already populated – skipping training.")
else:
    print("models/ directory not found or empty – proceeding with training.")

models/ directory not found or empty – proceeding with training.


## 3 – Fit vectorizers

In [4]:
if not MODELS_EXIST:
    print("Fitting CountVectorizer ...")
    count_vec = fit_count_vectorizer(train_df, ngram_range=(1, 1))
    print(f"  vocab size: {len(count_vec.vocabulary_):,}")

    print("Fitting TF-IDF Vectorizer ...")
    tfidf_vec = fit_tfidf_vectorizer(train_df, ngram_range=(1, 2), max_features=50_000)
    print(f"  vocab size: {len(tfidf_vec.vocabulary_):,}")
else:
    count_vec = joblib.load(f"{MODELS_DIR}/count_vectorizer.pkl")
    tfidf_vec = joblib.load(f"{MODELS_DIR}/tfidf_vectorizer.pkl")
    print("Loaded vectorizers from disk.")

Fitting CountVectorizer ...
  vocab size: 74,825
Fitting TF-IDF Vectorizer ...
  vocab size: 50,000


## 4 – Build feature matrices

In [5]:
if not MODELS_EXIST:
    print("Building baseline BoW features ...")
    X_tr_bow = get_bow_features(train_df, count_vec)
    X_val_bow = get_bow_features(val_df, count_vec)
    X_te_bow  = get_bow_features(test_df, count_vec)

    print("Building improved (all) features ...")
    X_tr_all  = build_all_features(train_df, count_vec, tfidf_vec)
    X_val_all = build_all_features(val_df,   count_vec, tfidf_vec)
    X_te_all  = build_all_features(test_df,  count_vec, tfidf_vec)

    print("X_tr_bow.shape :", X_tr_bow.shape)
    print("X_tr_all.shape :", X_tr_all.shape)

Building baseline BoW features ...
Building improved (all) features ...
X_tr_bow.shape : (291897, 149650)
X_tr_all.shape : (291897, 149656)


## 5 – Train baseline model (Logistic Regression on BoW)

In [6]:
if not MODELS_EXIST:
    y_train = train_df["is_duplicate"].values

    print("Training baseline Logistic Regression ...")
    lr_baseline = sklearn.linear_model.LogisticRegression(
        solver="liblinear", random_state=123, max_iter=200
    )
    lr_baseline.fit(X_tr_bow, y_train)

    evaluate_model(lr_baseline, X_tr_bow,  train_df["is_duplicate"].values, "Baseline – Train")
    evaluate_model(lr_baseline, X_val_bow, val_df["is_duplicate"].values,   "Baseline – Val  ")
    evaluate_model(lr_baseline, X_te_bow,  test_df["is_duplicate"].values,  "Baseline – Test ")

Training baseline Logistic Regression ...
Baseline – Train: ROC-AUC=0.8899 | Precision=0.7820 | Recall=0.6867
Baseline – Val  : ROC-AUC=0.8046 | Precision=0.6774 | Recall=0.6106
Baseline – Test : ROC-AUC=0.8137 | Precision=0.6952 | Recall=0.6186


## 6 – Train improved model (Logistic Regression on all features)

In [7]:
if not MODELS_EXIST:
    print("Training improved Logistic Regression ...")
    lr_improved = sklearn.linear_model.LogisticRegression(
        solver="liblinear", C=1.0, random_state=123, max_iter=200
    )
    lr_improved.fit(X_tr_all, y_train)

    evaluate_model(lr_improved, X_tr_all,  train_df["is_duplicate"].values, "Improved – Train")
    evaluate_model(lr_improved, X_val_all, val_df["is_duplicate"].values,   "Improved – Val  ")
    evaluate_model(lr_improved, X_te_all,  test_df["is_duplicate"].values,  "Improved – Test ")

Training improved Logistic Regression ...
Improved – Train: ROC-AUC=0.9158 | Precision=0.7893 | Recall=0.7583
Improved – Val  : ROC-AUC=0.8564 | Precision=0.7139 | Recall=0.6780
Improved – Test : ROC-AUC=0.8604 | Precision=0.7179 | Recall=0.6790


## 7 – Save everything to disk

In [8]:
if not MODELS_EXIST:
    os.makedirs(MODELS_DIR, exist_ok=True)

    save_model(count_vec,   f"{MODELS_DIR}/count_vectorizer.pkl")
    save_model(tfidf_vec,   f"{MODELS_DIR}/tfidf_vectorizer.pkl")
    save_model(lr_baseline, f"{MODELS_DIR}/lr_baseline.pkl")
    save_model(lr_improved, f"{MODELS_DIR}/lr_improved.pkl")

    print("All models saved.")
else:
    print("Skipped – models already exist on disk.")

All models saved.
